# TinySLM — intenções v02: descarregar e inspecionar

**Notebook autónomo para Kaggle.** Descarrega o pacote diretamente do GitHub público, confirma a integridade e apresenta as perguntas para revisão. Não carrega o checkpoint, não treina e não consome chamadas Codex.

No Kaggle, ativa **Internet** em *Notebook options* antes de executar. Se a Internet estiver indisponível, adiciona o ZIP publicado no GitHub em **Add Input**.

O CSV contém **rótulos candidatos automáticos**, não verificações humanas. A revisão é necessária antes do refinamento do modelo C2.


In [ ]:
from pathlib import Path
import csv, hashlib, io, json, random, urllib.request, zipfile
from collections import Counter, defaultdict

BASE = Path('/kaggle/working/tinyslm_intents_v02')
BASE.mkdir(parents=True, exist_ok=True)
ZIP = BASE / 'TinySLM_C2_intents_candidates_v02.zip'
URL = ('https://raw.githubusercontent.com/smartlearningci/tinyLLM/main/'
       'data/intent_candidates/TinySLM_C2_intents_candidates_v02.zip')
EXPECTED_SHA256 = '554d86397d2a488de90d788d73dc868392e1c269f3d9b6c9820b39c8204135d1'

if not ZIP.exists():
    inputs = list(Path('/kaggle/input').rglob(ZIP.name))
    if len(inputs) == 1:
        import shutil
        shutil.copyfile(inputs[0], ZIP)
    elif inputs:
        raise RuntimeError(f'Mais do que um ZIP encontrado: {inputs}')
    else:
        print('A descarregar do GitHub; ativa Internet nas opções do Kaggle...')
        urllib.request.urlretrieve(URL, ZIP)

actual = hashlib.sha256(ZIP.read_bytes()).hexdigest()
assert actual == EXPECTED_SHA256, f'SHA256 inesperado: {actual}'
with zipfile.ZipFile(ZIP) as archive:
    assert archive.testzip() is None, 'ZIP danificado'
    required = {f'intent_dataset_v02/{name}' for name in
                ('train.jsonl','validation.jsonl','test.jsonl','review.jsonl',
                 'review_sample_v02.csv','build_report.json')}
    assert required.issubset(archive.namelist()), 'Ficheiros em falta no ZIP'
    archive.extractall(BASE)
DATA = BASE/'intent_dataset_v02'
print('Dataset preparado em:', DATA)
print('SHA256:', actual)


In [ ]:
report = json.loads((DATA/'build_report.json').read_text())
print(json.dumps(report, indent=2, ensure_ascii=False))
assert report['status'] == 'HIGH_PRECISION_WEAK_CANDIDATES_NOT_YET_HUMAN_VERIFIED'


## Amostra estratificada para revisão

Preenche `decision_accept_correct_reject` com `accept`, `correct` ou `reject`.
Se escolheres `correct`, preenche também `human_intent` e `human_operation`.
O ficheiro pode ser descarregado em **Output** após o *Save Version*.


In [ ]:
with (DATA/'review_sample_v02.csv').open(encoding='utf-8', newline='') as stream:
    review = list(csv.DictReader(stream))
print('Linhas:', len(review))
print('Operações:', Counter((r['candidate_intent'],r['candidate_operation']) for r in review))
for i, row in enumerate(review[:20], 1):
    print(f"{i:02d} | {row['candidate_intent']}/{row['candidate_operation']} | {row['question']}")


## Exploração de perguntas por classe

Estas perguntas são exemplos de *weak labels* e servem para diagnóstico. Não representam uma medição independente de precisão.


In [ ]:
by_operation = defaultdict(list)
with (DATA/'train.jsonl').open(encoding='utf-8') as stream:
    for line in stream:
        row=json.loads(line)
        key=(row['target']['intent'],row['target']['operation'])
        by_operation[key].append(row['question'])
rng=random.Random(42)
for key, questions in sorted(by_operation.items()):
    print('\n', '/'.join(key), '—',len(questions))
    for q in rng.sample(questions, min(3,len(questions))):
        print(' •', q)


## Próxima fase: rotulagem por professor Codex

O pipeline anterior no GitHub (`distill.py` + `economico.py`) foi implementado para executar o **Codex CLI autenticado no computador local**, com SQLite, reservas de trabalho, JSON estruturado e retoma segura. O Kaggle **não** disponibiliza automaticamente essa sessão de autenticação. Assim, este notebook prepara e inspeciona o dataset; a anotação automática em paralelo será um processo local separado.

Antes de executar em escala: começar com 100 perguntas diversas, validar contra uma pequena amostra humana cega, registar desacordos, e só então aumentar o volume. Não utilizar os próprios rótulos do professor como única avaliação.
